# 00 — Kurulum ve Kontrol

Bu notebook Colab ortamını hazırlar: GPU kontrolü, Drive bağlantısı, repo klonu, bağımlılık kurulumu ve duman testi (BM25 ile küçük bir erişim denemesi).

**Başlamadan önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU** seçin.

Hücreleri sırayla çalıştırın. Bir hücre hata verirse önce o hatayı çözün; sonraki hücreler öncekilere bağımlıdır.

In [1]:
# 1) GPU ve ortam kontrolü
import sys
print('Python:', sys.version)
!nvidia-smi

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Tue Nov 10 16:52:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |             

In [2]:
# 2) Google Drive'ı bağla (sonuçlar ve model önbelleği burada tutulur)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 3) Repoyu klonla (varsa güncelle)
import os
REPO_URL = 'https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git'
BRANCH = 'RAG-dev'
REPO_DIR = '/content/RAG-proje'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
!git log --oneline -3 || echo 'henüz commit yok'

Cloning into '/content/RAG-proje'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 62 (delta 10), reused 61 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 44.51 KiB | 7.42 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/RAG-proje
01c3462 (HEAD -> RAG-dev, origin/RAG-dev) fix(colab): UTF-8 BOM, repo URL, trmteb model adi ve ornek veri dogrulama duzeltmeleri
f4a7882 (origin/main, origin/HEAD) feat: proje iskeleti, Colab notebooklari ve deney altyapisi


In [4]:
# 4) Bağımlılıkları kur (Colab'da torch hazır gelir; requirements bunu korur)
!pip install -q -r requirements.txt
print('kurulum tamam')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
kurulum tamam


In [5]:
# 5) Konfigürasyonu oku ve özetle
import sys
sys.path.insert(0, '/content/RAG-proje')
from src.config import load_config

cfg = load_config()
print('Deney:', cfg['deney']['ad'])
print('Tohumlar:', cfg['deney']['tohumlar'])
print('Modeller:')
for m in cfg['modeller']:
    print(' -', m['ad'], '->', m.get('hf'))

Deney: ana-karsilastirma
Tohumlar: [42, 43, 44, 45, 46]
Modeller:
 - minilm -> sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
 - e5-base -> intfloat/multilingual-e5-base
 - bge-m3 -> BAAI/bge-m3
 - qwen3-0.6b -> Qwen/Qwen3-Embedding-0.6B
 - tr-model -> trmteb/turkish-embedding-model
 - bm25 -> None


In [6]:
# 6) Duman testi: yönetmeliği parçala + BM25 ile örnek sorgu
# (model indirmez, CPU ile saniyeler sürer)
from src.data import korpus_yukle, yonetmelik_parcala
from src.retrieval import Bm25Retriever

metin = korpus_yukle(cfg['veri']['korpus'])
chunklar = yonetmelik_parcala(
    metin,
    boyut=cfg['parcalama']['boyut_karakter'],
    ortusme=cfg['parcalama']['ortusme_karakter'],
)
print(f'Chunk sayısı: {len(chunklar)}')

bm25 = Bm25Retriever(chunklar)
sorgu = 'Öğrencinin mezuniyet koşulları nelerdir?'
for chunk_id, skor in bm25.retrieve(sorgu, k=3):
    print(f'{skor:8.2f}  {chunk_id}')

# 7) Birim testleri doğrula
!python -m pytest tests/ -q
print('\nOrtam hazır. Sıradaki adım: 01_veri_hazirlama.ipynb')

Chunk sayısı: 82
    3.81  MADDE-37-3
    3.58  MADDE-20-2
    3.15  MADDE-34-5

.......                                                                  [100%]
7 passed in 0.04s

Ortam hazır. Sıradaki adım: 01_veri_hazirlama.ipynb


## Sorun giderme

- **GPU yok görünüyorsa:** Çalışma zamanı türünü T4 GPU olarak değiştirip oturumu yeniden başlatın.
- **pip çakışması:** Hücreyi bir kez daha çalıştırın; çözülmezse hata çıktısını ekip ile paylaşın (kendi başınıza sürüm sabitlerini değiştirmeyin, `requirements.txt` üzerinden PR açın).
- **Drive dolu hatası:** Model önbelleğini Drive'a değil oturum diskine yazın; sonuç CSV'leri küçüktür, sorun çıkarmaz.